In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import os

In [2]:
#load dataset
df = pd.read_csv('data/car_price_dataset.csv')
print(f"Dataset  Shape: {df.shape}")

df.head()

Dataset  Shape: (9788, 17)


,Unnamed: 0,Brand,Model,YOM,Engine (cc),Gear,Fuel Type,Millage(KM),Town,Date,Leasing,Condition,AIR CONDITION,POWER STEERING,POWER MIRROR,POWER WINDOW,Price
0,0,AUDI,A1,2016,990.0,Automatic,Petrol,99000.0,Gampaha,2025-02-05,No Leasing,USED,Available,Available,Available,Available,92.5
1,1,AUDI,A1,2017,1000.0,Automatic,Petrol,88000.0,Colombo,2025-01-14,No Leasing,USED,Available,Available,Available,Available,97.0
2,2,AUDI,A1,2018,1000.0,Automatic,Petrol,77000.0,Dehiwala-Mount-Lavinia,2025-01-23,No Leasing,USED,Available,Available,Available,Available,98.5
3,3,AUDI,A1,2017,1000.0,Automatic,Petrol,88000.0,Negombo,2024-12-21,No Leasing,USED,Available,Available,Available,Available,107.0
4,4,AUDI,A1,2017,1000.0,Automatic,Petrol,88000.0,Colombo,2024-12-21,No Leasing,USED,Available,Available,Available,Available,99.5


In [ ]:
# Dataset Info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9788 entries, 0 to 9787
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Unnamed: 0      9788 non-null   int64  
 1   Brand           9788 non-null   object 
 2   Model           9788 non-null   object 
 3   YOM             9788 non-null   int64  
 4   Engine (cc)     9788 non-null   float64
 5   Gear            9788 non-null   object 
 6   Fuel Type       9788 non-null   object 
 7   Millage(KM)     9788 non-null   float64
 8   Town            9788 non-null   object 
 9   Date            9788 non-null   object 
 10  Leasing         9788 non-null   object 
 11  Condition       9788 non-null   object 
 12  AIR CONDITION   9788 non-null   object 
 13  POWER STEERING  9788 non-null   object 
 14  POWER MIRROR    9788 non-null   object 
 15  POWER WINDOW    9788 non-null   object 
 16  Price           9788 non-null   float64
dtypes: float64(3), int64(2), object(1

In [3]:
# Summary Statistics
df.describe()

,Unnamed: 0,YOM,Engine (cc),Millage(KM),Price
count,9788.000000,9788.000000,9788.000000,9788.000000,9788.000000
mean,4893.758991,2006.759093,1260.442358,200649.979567,53.460954
std,2826.028988,9.831574,403.769059,108147.314860,43.615277
min,0.000000,1956.000000,573.000000,11000.000000,2.650000
25%,2446.750000,2001.000000,1000.000000,110000.000000,26.250000
50%,4893.500000,2009.000000,1300.000000,176000.000000,43.000000
75%,7341.250000,2015.000000,1500.000000,264000.000000,69.750000
max,9788.000000,2024.000000,4800.000000,759000.000000,790.000000


In [ ]:
# Check for missing values
print("Missing Values:\n", df.isnull().sum())

Missing Values:
 Unnamed: 0        0
Brand             0
Model             0
YOM               0
Engine (cc)       0
Gear              0
Fuel Type         0
Millage(KM)       0
Town              0
Date              0
Leasing           0
Condition         0
AIR CONDITION     0
POWER STEERING    0
POWER MIRROR      0
POWER WINDOW      0
Price             0
dtype: int64


In [13]:
X = df.drop(columns=['Unnamed: 0','Price', 'Date'], errors='ignore')
y = df['Price']

In [11]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 1. Define Column Groups
categorical_cols_label = ['Brand', 'Model', 'Town']
categorical_cols_onehot = [col for col in X.select_dtypes(include='object').columns if col not in categorical_cols_label]
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Label Encoded: {categorical_cols_label}")
print(f"One-Hot Encoded: {categorical_cols_onehot}")
print(f"Numeric: {numeric_cols}")




In [6]:
# 2. Label Encoding 
label_encoders = {}
X_train_le = X_train.copy()
X_test_le = X_test.copy()

for col in categorical_cols_label:
    if col in X_train.columns:
        le = LabelEncoder()
        # Convert to string to ensure consistency (e.g. if some models are numbers)
        # Use UPPER CASE as per our fix
        X_train_le[col] = le.fit_transform(X_train[col].astype(str).str.strip().str.upper())
        
        # Handle unseen labels in test set: map to a default or first class (0)
        X_test_le[col] = X_test[col].astype(str).str.strip().str.upper().apply(lambda x: le.transform([x])[0] if x in le.classes_ else 0)
        
        label_encoders[col] = le



In [7]:
# 3. Pipeline / ColumnTransformer
# OneHotEncoder handles the rest of categorical columns
# StandardScaler for numeric
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat_onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols_onehot)
    ],
    remainder='passthrough'
)

# Fit preprocessor on Train
X_train_processed = preprocessor.fit_transform(X_train_le)
X_test_processed = preprocessor.transform(X_test_le)

print(f"Processed Train Shape: {X_train_processed.shape}")


Processed Train Shape: (7830, 25)


In [8]:
# 4. Train Models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'SVR': SVR()
}

results = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_processed, y_train)
    y_pred = model.predict(X_test_processed)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
    print(f"{name} -> MAE: {mae:.2f}, RMSE: {rmse:.2f}, R2: {r2:.4f}")




Training Linear Regression...
Linear Regression -> MAE: 16.84, RMSE: 28.24, R2: 0.5556
Training Decision Tree...
Decision Tree -> MAE: 6.81, RMSE: 20.29, R2: 0.7706
Training Random Forest...
Random Forest -> MAE: 5.47, RMSE: 14.87, R2: 0.8768
Training SVR...
SVR -> MAE: 26.76, RMSE: 43.69, R2: -0.0636


In [10]:
# 5. Save Artifacts (Random Forest as Best Model)
best_model = models['Random Forest']

# Ensure directory
os.makedirs('model', exist_ok=True)

# Save
joblib.dump(best_model, 'model/vehicle_price_model.pkl')
joblib.dump(preprocessor, 'model/preprocessor.pkl')
joblib.dump(label_encoders, 'model/label_encoders.pkl')

print("Artifacts saved to model/")

Artifacts saved to model/
